# Notebook 8 — Emissions Tracking & Carbon Policy

Every generator that burns fuel emits CO2. This notebook explores how the simulator tracks emissions and how carbon pricing affects the dispatch.

Topics:
1. Per-generator emissions accounting
2. Carbon intensity (gCO2/kWh) — the key policy metric
3. Emissions breakdown by technology
4. How CO2 price drives fuel-switching
5. Carbon intensity vs renewable penetration

**Runtime**: ~60 seconds

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from energy_sim.models import TimeGrid, LoadProfile
from energy_sim.generators import build_generators, CarbonPriceModel
from energy_sim.dispatch import dispatch_year
from energy_sim.config import (
    ITALIAN_MIX, GAS_SCENARIOS, CO2_SCENARIOS, P_PEAK_GW,
    QUARTERS_PER_DAY, WEEKDAY_LOAD_FACTORS, HOLIDAY_LOAD_FACTOR,
    ITALIAN_HOLIDAYS_DOY, DEFAULT_LOAD_NOISE_SIGMA,
)
from energy_sim.simulation import run_monte_carlo, sweep_technology

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

## 1. How emissions are computed

For each generator at each timestep:

$$E_{i,t} = P_{i,t} \times P_{\text{BASE}} \times \Delta t \times 1000 \times \frac{e_f}{\eta}$$

Where:
- $P_{i,t}$ = dispatched power (p.u.)
- $P_{\text{BASE}}$ = 60 GW
- $\Delta t$ = 0.25 h
- $e_f$ = emission factor (tCO2/MWh_th)
- $\eta$ = efficiency

Result is in **tonnes CO2 per quarter-hour**.

In [ ]:
# Run a single dispatch
tg = TimeGrid()
tg.set_holiday_calendar(ITALIAN_HOLIDAYS_DOY)
lp = LoadProfile(tg)
lp.set_weekday_factors(WEEKDAY_LOAD_FACTORS)
lp.set_holiday_factor(HOLIDAY_LOAD_FACTOR)

rng = np.random.default_rng(42)
co2 = CarbonPriceModel()
gens = build_generators(ITALIAN_MIX, GAS_SCENARIOS['base'])
for g in gens:
    g.prepare_run(tg, rng, co2)
load = lp.generate(rng, noise_sigma=DEFAULT_LOAD_NOISE_SIGMA)

result = dispatch_year(gens, load)

# Annual emissions per generator
print(f"{'Generator':<14s} {'EF (tCO2/MWh_th)':>18s} {'Efficiency':>10s} {'Annual emissions':>16s}")
print("-" * 65)
total = 0
for i, name in enumerate(result.gen_names):
    annual = result.emissions[i].sum()
    total += annual
    ef = gens[i].emission_factor
    eff = gens[i].efficiency
    print(f"{name:<14s} {ef:18.2f} {eff:10.2f} {annual/1e6:13.3f} Mt CO2")

total_energy_twh = result.power.sum() * P_PEAK_GW * 0.25 / 1e3
ci = total / 1e6 * 1e6 / (total_energy_twh * 1e6)  # gCO2/kWh
print(f"\nTotal: {total/1e6:.2f} Mt CO2")
print(f"Energy served: {total_energy_twh:.1f} TWh")
print(f"Carbon intensity: {ci:.0f} gCO2/kWh")

In [ ]:
# Emissions breakdown pie chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_map = {'gas': 'steelblue', 'solar': 'gold', 'wind': 'teal',
              'hydro_mustrun': 'royalblue', 'nuclear': 'purple', 'coal': 'brown'}

# Only show emitters
emitters = [(name, result.emissions[i].sum())
            for i, name in enumerate(result.gen_names)
            if result.emissions[i].sum() > 0]

if emitters:
    names_e, values_e = zip(*emitters)
    axes[0].pie(values_e,
                labels=[f"{n}\n{v/1e6:.2f} Mt" for n, v in emitters],
                colors=[colors_map.get(n, 'gray') for n in names_e],
                autopct='%1.0f%%')
    axes[0].set_title('CO2 emissions by technology')

# Monthly emissions
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly_emis = [result.emissions.sum(axis=0)[tg.month == m].sum() / 1e6 for m in range(1, 13)]
axes[1].bar(month_names, monthly_emis, color='coral')
axes[1].set_ylabel('Mt CO2')
axes[1].set_title('Monthly CO2 emissions')

plt.tight_layout()
plt.show()

## 2. How CO2 price affects fuel-switching

Run the same mix under different CO2 price scenarios and observe how emissions change:

In [ ]:
from copy import deepcopy

# Add coal to make fuel-switching visible
mix_with_coal = deepcopy(ITALIAN_MIX)
mix_with_coal['coal']['capacity_gw'] = 10.0

co2_results = {}
for name, params in CO2_SCENARIOS.items():
    mc = run_monte_carlo(
        mix_with_coal, GAS_SCENARIOS['base'],
        co2_scenario=params,
        n_runs=10, seed=42,
    )
    co2_results[name] = mc
    print(f"CO2 {name:>5s} (mu={params['mu']:3.0f}): "
          f"price={mc['avg_price'].mean():.1f} EUR/MWh, "
          f"CI={mc['carbon_intensity'].mean():.0f} gCO2/kWh, "
          f"emissions={mc['total_emissions'].mean()/1e6:.2f} Mt")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

co2_mus = [CO2_SCENARIOS[n]['mu'] for n in co2_results]
prices = [co2_results[n]['avg_price'].mean() for n in co2_results]
cis = [co2_results[n]['carbon_intensity'].mean() for n in co2_results]
emis = [co2_results[n]['total_emissions'].mean()/1e6 for n in co2_results]

axes[0].bar(list(co2_results.keys()), prices, color='steelblue')
axes[0].set_ylabel('Electricity price (EUR/MWh)')
axes[0].set_title('Price vs CO2 scenario')

axes[1].bar(list(co2_results.keys()), cis, color='coral')
axes[1].set_ylabel('Carbon intensity (gCO2/kWh)')
axes[1].set_title('Carbon intensity vs CO2 scenario')

# Per-tech emissions comparison
x = np.arange(len(co2_results))
bottom = np.zeros(len(co2_results))
for tech in ['gas', 'coal']:
    vals = [co2_results[n]['emissions_by_tech'].get(tech, np.array([0])).mean()/1e6
            for n in co2_results]
    axes[2].bar(x, vals, bottom=bottom, label=tech,
                color=colors_map.get(tech, 'gray'))
    bottom += vals
axes[2].set_xticks(x)
axes[2].set_xticklabels(list(co2_results.keys()))
axes[2].set_ylabel('Annual emissions (Mt CO2)')
axes[2].set_title('Emissions by tech vs CO2 scenario')
axes[2].legend()

plt.tight_layout()
plt.show()

## 3. Carbon intensity vs renewable penetration

In [ ]:
# Sweep nuclear to see its decarbonization impact
nuc_sweep = sweep_technology(
    ITALIAN_MIX, 'nuclear',
    np.array([0, 5, 10, 15, 20, 25]),
    GAS_SCENARIOS['base'],
    n_runs=10, seed=42,
)

In [ ]:
pcts = [r['pct'] for r in nuc_sweep]
ci_values = [r['mean_carbon_intensity'] for r in nuc_sweep]
emis_values = [r['mean_emissions'] / 1e6 for r in nuc_sweep]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(pcts, ci_values, 'o-', color='purple', lw=2, ms=8)
axes[0].set_xlabel('Nuclear penetration (%)')
axes[0].set_ylabel('Carbon intensity (gCO2/kWh)')
axes[0].set_title('Carbon intensity vs nuclear penetration')

axes[1].plot(pcts, emis_values, 'o-', color='coral', lw=2, ms=8)
axes[1].set_xlabel('Nuclear penetration (%)')
axes[1].set_ylabel('Annual emissions (Mt CO2)')
axes[1].set_title('Total emissions vs nuclear penetration')

plt.tight_layout()
plt.show()

# Marginal abatement
if len(pcts) > 1:
    dE = np.diff(emis_values)
    dP = np.diff(pcts)
    for i in range(len(dE)):
        print(f"  {pcts[i]:.0f}% → {pcts[i+1]:.0f}%: "
              f"{dE[i]:+.2f} Mt CO2 per +{dP[i]:.0f}%pt nuclear")

## Key Takeaways

1. **Emissions are computed per-generator, per-timestep** — full granularity allows both annual totals and hourly patterns.
2. **Gas dominates emissions** in the Italian mix (no coal). Coal has ~70% higher CO2 per MWh_th than gas.
3. **CO2 price is the policy lever**: high CO2 prices flip the merit order from coal to gas, reducing emissions — this is the EU ETS mechanism.
4. **Nuclear displaces gas** (the marginal emitter), so each percentage point of nuclear reduces emissions roughly linearly.
5. The **carbon intensity** metric (gCO2/kWh) is the standard for comparing electricity systems internationally.

**Next notebook**: [09 — Interconnections](./09_interconnections.ipynb) — cross-border electricity trading.